In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np

# Load the dataset
df = pd.read_csv("ER Wait Time Dataset.csv")

# Drop unwanted columns
drop_cols = [
    'Visit ID', 'Patient ID', 'Hospital ID', 'Hospital Name',
    'Visit Date',  # <- drop Visit Date to avoid string errors
    'Time to Registration (min)',
    'Time to Triage (min)',
    'Time to Medical Professional (min)', "Patient Outcome","Patient Satisfaction"
]
df.drop(columns=drop_cols, inplace=True)

# Convert Urgency Level to numeric (Low = 1, Medium = 2, High = 3)
urgency_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['Urgency Level'] = df['Urgency Level'].map(urgency_map)

# Split features and target
X = df.drop(columns=['Total Wait Time (min)'])
y = df['Total Wait Time (min)']

# One-hot encode categorical columns (excluding Urgency Level)
categorical_cols = ['Region', 'Day of Week', 'Season', 'Time of Day']

# Define column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'  # keep numeric columns including Urgency Level
)

# Create model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

# Save model
joblib.dump(model_pipeline, "waitwise_model_pipeline.joblib")


RMSE: 17.99
R² Score: 0.9304


['waitwise_model_pipeline.joblib']

In [8]:
import joblib

# Path to your joblib model file
model_path = "waitwise_model_pipeline.joblib"

# Load the pipeline
pipeline = joblib.load(model_path)

# If it's a sklearn.pipeline.Pipeline, inspect its named steps
if hasattr(pipeline, 'named_steps'):
    print("Pipeline steps:")
    for step_name, step_obj in pipeline.named_steps.items():
        print(f"- {step_name}: {type(step_obj)}")
else:
    print("Loaded object is not a Pipeline.")


from sklearn.compose import ColumnTransformer

preprocessor = pipeline.named_steps.get('preprocessor')  # Adjust name if different
if isinstance(preprocessor, ColumnTransformer):
    for name, transformer, columns in preprocessor.transformers_:
        print(f"\nTransformer: {name}")
        print(f"  Columns: {columns}")
        print(f"  Transformer type: {type(transformer)}")


Pipeline steps:
- preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
- regressor: <class 'sklearn.ensemble._forest.RandomForestRegressor'>

Transformer: cat
  Columns: ['Region', 'Day of Week', 'Season', 'Time of Day']
  Transformer type: <class 'sklearn.preprocessing._encoders.OneHotEncoder'>

Transformer: remainder
  Columns: [4, 5, 6, 7]
  Transformer type: <class 'sklearn.preprocessing._function_transformer.FunctionTransformer'>


In [9]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid for the Random Forest
param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [5, 7, 10],
    'regressor__min_samples_split': [5, 10],
    'regressor__min_samples_leaf': [2, 4],
    'regressor__max_features': ['sqrt']
}

# Wrap the current pipeline in GridSearchCV
grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Run grid search
grid_search.fit(X_train, y_train)

# Best model from the grid search
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

# Evaluate on test set
y_pred_best = best_model.predict(X_test)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

best_params, rmse_best, r2_best


Fitting 3 folds for each of 24 candidates, totalling 72 fits


({'regressor__max_depth': 10,
  'regressor__max_features': 'sqrt',
  'regressor__min_samples_leaf': 2,
  'regressor__min_samples_split': 5,
  'regressor__n_estimators': 100},
 20.846230531179202,
 0.9065882269607596)

Modified Training


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define the best parameters manually
best_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

# ColumnTransformer for categorical features
categorical_cols = ['Region', 'Day of Week', 'Season', 'Time of Day']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Final pipeline using best model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', best_rf)
])

# Fit the model
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Updated Model - RMSE: {rmse:.2f} | R²: {r2:.4f}")

import joblib
joblib.dump(model_pipeline, "waitwise_model_pipeline_tuned.joblib")


Updated Model - RMSE: 20.85 | R²: 0.9066


['waitwise_model_pipeline_tuned.joblib']